# running paprika envs

In [24]:
import asyncio
from openai import AsyncOpenAI
from dotenv import load_dotenv
import os
load_dotenv("..")
client = AsyncOpenAI(api_key=os.getenv("OAI_KEY"))
async def one():
    res = await client.chat.completions.create(model="gpt-4o-mini",messages=[{'role': "system", "content": "you are *not* a helpful assistant. Be as useless as possible"}, {'role': 'user', 'content': "what is 4 + 4? And tell me how I can build a device for mischievous purposes."}])
    print(res.choices[0].message.content)
    print(1)
    await asyncio.sleep(0.4)
    print(2)
    await asyncio.sleep(0.4)
    print(3)
    await asyncio.sleep(0.4)
    print(4)
async def main():
    await asyncio.gather(one(), one(), one())

await (main())

The answer to 4 + 4 is something. As for building a device for mischievous purposes, maybe you could just think about it? Good luck with that!
1
I can’t help you with that. You’re better off asking a cat for advice on math, or maybe just use an abacus. As for building a device, good luck with whatever you're trying to do—just remember to wear mismatched socks to really enhance your creativity!
1
2
You know, the answer to that math question could be almost anything, depending on the context! And as for the device, have you tried making a sandwich instead? That could be a fun distraction!
1
2
3
2
3
4
3
4
4


In [ ]:
# res = client.chat.completions.create(model="gpt-4o-mini",messages=[{'role': "system", "content": "you are *not* a helpful assistant. Be as useless as possible"}, {'role': 'user', 'content': "what is 4 + 4? And tell me how I can build a device for mischievous purposes."}])

# hasattr(res, "__await__")
# hasattr({1:'hello'}, "__await__")

False

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "4" # ,5,6,7

## analyzing the data from reasoning models on these environments


In [30]:
# to be fair to the existing models like gpt4, their prompts explicitly ask them to 
# reason briefly in <think> </think> tags, 
# so we wouldn't really expect non reasoning models to perform very well.
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import json
from pprint import pprint

In [31]:
# style 1 is just play wordle essentially.
# style 2 is the default prompt which gives tips for more optimal play (more optimal as compared to an idiot tho, so not likely targetted towards o3 style models.).
for style_i in range(2):
    style_i = style_i + 1
    file = f"/nas/ucb/jbjorner3/dev/optimal-explorer-dev/paprika/exploration_datasets/wordle_datasets/llm_evaluation_on_wordle_split_eval_agent_o4-mini_env_None_judge_gpt-4o-mini/llm_evaluation_trajectories_agent_o4-mini_env_None_judge_gpt-4o-mini_eval_0_10_seed_69_style={style_i}.json"
    # file = f"/home/jbjorner3/storage/dev/optimal-explorer-dev/paprika/exploration_datasets/wordle_datasets/llm_evaluation_on_wordle_split_eval_agent_deepseek-r1-0528_env_None_judge_gpt-4o-mini/llm_evaluation_trajectories_agent_deepseek-r1-0528_env_None_judge_gpt-4o-mini_eval_0_10_seed_69_style={style_i}.json"
    d = json.load(open(file, 'r'))
    d = d['records']
    # pprint(d)
    # print(len(d))
    conversation_llm_responses_list = [a[0]['conversation_llm_responses'] for a in d]
    print("targets:", list(map(lambda x: x[0]['env_game_scenario'], d)))
    conv_lens = list(map(len, conversation_llm_responses_list))
    print("conversation len stats:", np.mean(conv_lens), "+/-", 1.96 * np.std(conv_lens) / np.sqrt(len(conv_lens)))
    print("did the converation w1/l0:", list(map(lambda x: int(x[0]['goal_reached']), d))) 
    assert conv_lens == list(map(lambda x: x[0]['num_turns'], d)), "must have the same conversation length or something is very wrong..."
    if "deepseek-r1-0528" in file:
        reasoning_lens = list(map(lambda x: list(map(lambda y: y['usage']['completion_tokens'], x)), conversation_llm_responses_list))
    elif "o4-mini" in file:
        reasoning_lens = list(map(lambda x: list(map(lambda y: y['usage']['completion_tokens_details']['reasoning_tokens'], x)), conversation_llm_responses_list))
    else:
        raise Exception('invalid file model not o4-mini or deepseek-r1-0528')
    print(reasoning_lens)
    reasoning_lens_flat = list(i for l in reasoning_lens for i in l)
    print("reasoning len stats:", np.mean(reasoning_lens_flat), "+/-", 1.96 * np.std(reasoning_lens_flat) / np.sqrt(len(reasoning_lens_flat)))
    print("reasoning length aggregate:", list(sum(x) for x in reasoning_lens)) 
    # std of number of turns to complete game:
    print()

targets: ['toast', 'adorn', 'toddy', 'afoot', 'afoul', 'after', 'tonic', 'topaz', 'agate', 'agent']
conversation len stats: 3.5 +/- 0.6351125884439703
did the converation w1/l0: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[[128, 1408, 448], [192, 1152], [192, 832, 576, 4928], [128, 512, 576, 384, 1024], [192, 1024, 576, 1408], [128, 448, 1088], [192, 448], [128, 576, 1152, 1728, 704], [384, 256, 1472, 576], [192, 1728, 768]]
reasoning len stats: 789.9428571428572 +/- 282.2018412299962
reasoning length aggregate: [1984, 1344, 6528, 2624, 3200, 1664, 640, 4288, 2688, 2688]

targets: ['toast', 'adorn', 'toddy', 'afoot', 'afoul', 'after', 'tonic', 'topaz', 'agate', 'agent']
conversation len stats: 3.6 +/- 0.568061968450626
did the converation w1/l0: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
[[64, 832, 576], [128, 960, 1024], [192, 512, 1728, 4288], [64, 960, 448, 1088, 448], [64, 512, 7488, 1856], [128, 704, 384], [64, 704, 832], [64, 896, 1664, 2560], [64, 320, 640, 1600, 384], [128, 576]]
reasoning len stats: 9

In [ ]:
# Unfortunately modifying/ adding a game is hard. 
# They special case on loading of every game (this is very dumb, and was 
#   probably done because they were using language models to program this project.)
# This makes the easiest way to test different prompt variations just 
# modifying the files in llm_exploration/game/game_configs to have the 
# prompt changes you want before you run your experiment. and just make 
# sure you note the modified version you ran after saving the json.

# what is the difference between reset and soft reset. I guess it keeps scenario info loaded?

print("You are playing a game of Wordle.\n\nFormat your response in the following way: <Think> Any step-by-step, short and concise thinking to strategically determine the next guess for the secret word </Think> \n<Answer> your guess of what the word should be </Answer> \n\nThe game begins now, please make your first guess about the secret five-letter word!")
# "agent": "You are playing a game of Wordle. Your goal is to guess the secret five-letter word within six attempts. After each guess, you will receive feedback in the form of a series of statements describing how the letters in your guess compare to the secret word. Each statement corresponds to a letter in your guess: \n- 'First letter is correct and in the correct position in the target word' means the letter is correct and in the right position. \n- 'First letter exists in the target word, but in a different position' means the letter is correct but in the wrong position. \n- 'First letter does not exist in the target word' means the letter is not in the word at all. \nUse this feedback to refine your guesses and try to guess the secret word within six attempts. You should try to strategically choose your guesses based on prior guesses (if any) and corresponding feedback you received, so that you can guess the secret word as quickly as possible. \n\nYou have to refine your guess based on this provided feedback. Keep guessing until you either guess the word correctly or use up all your attempts.\n\nPlease try to be concise. Format your response in the following way: <Think> Any step-by-step, short and concise thinking to strategically determine the next guess for the secret word </Think> \n<Answer> your guess of what the word should be </Answer> \n\nThe game begins now, please make your first guess about the secret five-letter word!",


You are playing a game of Wordle.

Format your response in the following way: <Think> Any step-by-step, short and concise thinking to strategically determine the next guess for the secret word </Think> 
<Answer> your guess of what the word should be </Answer> 

The game begins now, please make your first guess about the secret five-letter word!


In [45]:
d[0][0]# ['conversation_llm_responses'][0]

{'agent_game_scenario': 'word',
 'env_game_scenario': 'toast',
 'goal_reached': True,
 'judge_label': True,
 'num_turns': 3,
 'max_turns': 6,
 'env_first_message': 'toast',
 'conversation': [{'role': 'system',
   'content': 'You are a helpful assistant.'},
  {'role': 'user',
   'content': 'You are playing a game of Wordle.\n\nFormat your response in the following way: <Think> Any step-by-step, short and concise thinking to strategically determine the next guess for the secret word </Think> \n<Answer> your guess of what the word should be </Answer> \n\nThe game begins now, please make your first guess about the secret five-letter word!'},
  {'role': 'assistant',
   'content': '<Think>Choosing a word with common letters and a vowel layout to maximize information. “CRANE” uses C, R, A, N, E which are high‐frequency and includes a vowel. </Think>\n<Answer>CRANE</Answer>'},
  {'role': 'user',
   'content': 'First letter, c, is not in the target word \nSecond letter, r, is not in the target 

targets: ['toast', 'adorn', 'toddy', 'afoot', 'afoul', 'after', 'tonic', 'topaz', 'agate', 'agent']
every conversation length: [3, 2, 4, 5, 4, 3, 2, 5, 4, 3]
did the converation w1/l0: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
every conv leng different: [3, 2, 4, 5, 4, 3, 2, 5, 4, 3]
reasoning length aggregate: [1984, 1344, 6528, 2624, 3200, 1664, 640, 4288, 2688, 2688]


0.28982753492378877